# S1 · 03 · Reto opcional: la misma pregunta, dos modelos
**Reto** (≈ 15 min, opcional): haces la misma pregunta con tool a **Gemini** y a **`gemma4:e2b`** y comparas si llamaron la tool y cuántos tokens gastaron.
Necesitas IA real (`COURSE_OFFLINE=0` en `.env` y *Restart*), Ollama con `gemma4:e2b` y tu clave de Gemini. Sin IA real corre, pero no mide nada.


## Preparación
El agente es el del Ejercicio 2 (`minutos_entre`), ya resuelto. `armar_agente(modelo)` lo arma con el modelo que le des, así podemos cambiar solo el modelo.


In [ ]:
from pydantic_ai import Agent
from pydantic_ai.messages import ToolCallPart
from utilidades.config import crear_modelo, describir_ruta
from utilidades.practica import preparar, revisar

settings, OFFLINE = preparar()


def minutos_entre(inicio: str, fin: str) -> int:
    """Minutos que hay entre dos horas del mismo día. `inicio` y `fin` en formato HH:MM (24 horas)."""
    h1, m1 = inicio.split(":")
    h2, m2 = fin.split(":")
    return (int(h2) * 60 + int(m2)) - (int(h1) * 60 + int(m1))


def armar_agente(modelo) -> Agent:
    return Agent(
        modelo,
        instructions="Responde en español, breve. Usa las herramientas para calcular.",
        tools=[minutos_entre],  # otra forma de registrar una tool: en la lista tools=[...]
    )


PREGUNTA = "¿Cuántos minutos hay entre 09:15 y 11:45?"
# `settings.model_copy(update=...)` copia tu configuración cambiando solo la ruta.
VARIANTES = [{"mode": "gemini"}, {"mode": "local", "ollama_model": "gemma4:e2b"}]


## Reto · ¿Cuál llamó la tool y cuánto gastó?
**Se apoya en:** Demo 2 · **Nivel:** reto. **Qué cambia:** en la demo había un solo modelo; aquí el agente es el mismo y lo mides con dos.

Completa `comparar(pregunta, variantes)`: para cada variante crea `cfg = settings.model_copy(update=cambios)`, un agente con `armar_agente(crear_modelo(cfg, OFFLINE))`, ejecútalo y añade una fila con:
- `modelo`: `describir_ruta(cfg, OFFLINE)`;
- `llamo_tool`: `True` si algún mensaje tiene un `ToolCallPart` (mira `r.all_messages()` y `m.parts`);
- `entrada` y `salida`: los tokens de `r.usage.input_tokens` y `r.usage.output_tokens`;
- `respuesta`: `r.output`.

Si un modelo falla (sin clave, sin cuota, Ollama apagado), su fila lleva `error` en lugar de métricas: atrapa la excepción por variante.


In [ ]:
async def comparar(pregunta: str, variantes: list[dict]) -> list[dict]:
    # TODO Reto: una fila por variante con modelo, llamo_tool, entrada, salida y respuesta (o error).
    return []


filas = await comparar(PREGUNTA, VARIANTES)
for fila in filas:
    print(fila)


Escribe tu conclusión en una frase: ¿cuál llamó la tool?, ¿cuál gastó menos tokens?, ¿qué **no** puedes concluir de una sola pregunta?


In [ ]:
CONCLUSION = ""  # TODO Reto: tu conclusión en una frase


In [ ]:
revisar(
    "Reto · una fila por modelo, con métricas o error",
    lambda: len(filas) == 2 and all({"llamo_tool", "entrada", "salida"} <= set(f) or "error" in f for f in filas),
    "Una fila por variante con llamo_tool, entrada y salida (o «error» si ese modelo falló)",
)
revisar("Reto · escribiste una conclusión", lambda: len(CONCLUSION.split()) >= 8, "Escribe una frase completa en CONCLUSION")
